# Car Price Prediction - Complete Analysis
## 12 Months Data from unegui.mn (Nov 2024 - Oct 2025)

This notebook provides a complete end-to-end analysis:
1. Data Loading (12 CSV files)
2. Deduplication
3. Data Cleaning
4. Feature Engineering
5. Exploratory Data Analysis (11 Questions)
6. Machine Learning Model Training
7. Price Prediction

**Note:** This notebook is optimized for Google Colab and local Jupyter

## Setup and Imports

In [ ]:
# Install required packages (uncomment for Colab)
# !pip install pandas numpy scikit-learn matplotlib seaborn tqdm xgboost lightgbm catboost

import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

print("✓ Setup complete")

## 1. Data Loading
Load all 12 months of car advertisement data

In [ ]:
from src.data.loader import CarDataLoader
from src.utils.logger import setup_logger

# Setup logging
setup_logger('analysis', level='INFO')

# Load data
loader = CarDataLoader()

# Get file summary first
print("File Summary:")
summary = loader.get_file_summary()
print(summary)
print(f"\nTotal files: {len(summary)}")
print(f"Total rows: {summary['row_count'].sum():,}")
print(f"Total size: {summary['file_size_mb'].sum():.2f} MB")

# Load all data
print("\nLoading all data...")
df_raw = loader.load_all_files(use_checkpoint=True)

print(f"\n✓ Loaded {len(df_raw):,} ads")
print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw['collection_date'].min()} to {df_raw['collection_date'].max()}")

## 2. Deduplication
Remove duplicate ads across months and track changes

In [ ]:
from src.data.deduplicator import AdDeduplicator

dedup = AdDeduplicator()

# Get duplicate summary
print("Duplicate Summary:")
dup_summary = dedup.get_duplicate_summary(df_raw)
for key, value in dup_summary.items():
    if key != 'top_duplicated_ids':
        print(f"{key}: {value}")

# Create time series tracking (for temporal analysis)
print("\nCreating time series tracking...")
df_tracking = dedup.create_time_series_tracking(df_raw)

# For ML, use latest version of each ad
print("\nDeduplicating for ML (keeping latest)...")
df_dedup = dedup.deduplicate_across_files(df_raw, strategy='keep_latest')

print(f"\n✓ Deduplicated: {len(df_dedup):,} unique ads")

## 3. Data Cleaning
Remove outliers and handle missing values

In [ ]:
from src.data.cleaner import DataCleaner

cleaner = DataCleaner()

print("Cleaning data...")
df_clean = cleaner.clean_data(df_dedup, remove_outliers=True, handle_missing=True)

# Show cleaning report
print("\nCleaning Report:")
report = cleaner.get_cleaning_report()
for key, value in report.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

print(f"\n✓ Cleaned data: {len(df_clean):,} ads")

## 4. Feature Engineering
Create features for machine learning

In [ ]:
from src.data.feature_engineering import FeatureEngineer

fe = FeatureEngineer()

print("Engineering features...")
df_featured = fe.engineer_features(df_clean)

print(f"\n✓ Feature engineering complete")
print(f"Shape: {df_featured.shape}")

# Show new features
original_cols = set(df_clean.columns)
new_cols = [c for c in df_featured.columns if c not in original_cols]
print(f"\nNew features created ({len(new_cols)}):")
for col in new_cols:
    print(f"  - {col}")

## 5. Exploratory Data Analysis (11 Questions)

In [ ]:
from src.analysis.eda_11_questions import EDAAnalyzer

analyzer = EDAAnalyzer(df_featured)

print("Running 11 EDA questions...")
results = analyzer.run_all_questions(save_results=True)

print("\n✓ Analysis complete! Results saved to outputs/reports/")

### Q1: Most Sold Brands and Marks

In [ ]:
q1 = results['q1']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top brands
brands = pd.Series(q1['top_brands'])
axes[0].barh(range(len(brands)), brands.values)
axes[0].set_yticks(range(len(brands)))
axes[0].set_yticklabels(brands.index)
axes[0].set_xlabel('Number of Ads')
axes[0].set_title('Top Brands')
axes[0].invert_yaxis()

# Top marks
marks = pd.Series(q1['top_marks']).head(10)
axes[1].barh(range(len(marks)), marks.values)
axes[1].set_yticks(range(len(marks)))
axes[1].set_yticklabels(marks.index)
axes[1].set_xlabel('Number of Ads')
axes[1].set_title('Top Marks')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Q4: Monthly Ad Volume

In [ ]:
if 'q4' in results and 'monthly_counts' in results['q4']:
    q4 = results['q4']
    monthly = pd.Series(q4['monthly_counts'])
    
    plt.figure(figsize=(14, 6))
    plt.plot(monthly.index, monthly.values, marker='o', linewidth=2)
    plt.xlabel('Month')
    plt.ylabel('Number of Ads')
    plt.title('Monthly Ad Volume')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Average monthly ads: {q4['avg_monthly_ads']:.0f}")
    print(f"Peak month: {q4['max_month']}")

## 6. Machine Learning - Price Prediction

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

# Prepare data for ML
print("Preparing data for machine learning...")

# Select features
feature_cols = ['car_age', 'mileage', 'mileage_per_year', 'is_automatic', 
                'is_4wd', 'is_brand_new', 'doors']

# Filter available features
feature_cols = [c for c in feature_cols if c in df_featured.columns]

# Add categorical features (encoded)
categorical_features = ['brand', 'type', 'transmission']
for cat in categorical_features:
    if cat in df_featured.columns:
        # Simple label encoding
        df_featured[f'{cat}_encoded'] = pd.Categorical(df_featured[cat]).codes
        feature_cols.append(f'{cat}_encoded')

# Prepare X and y
ml_df = df_featured.dropna(subset=feature_cols + ['price_in_mil'])
X = ml_df[feature_cols]
y = ml_df['price_in_mil']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"Features: {len(feature_cols)}")

In [ ]:
# Train Random Forest
print("\nTraining Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"\nRandom Forest Results:")
print(f"  R² Score: {r2_rf:.4f} ({r2_rf*100:.2f}% accuracy)")
print(f"  MAE: {mae_rf:.2f} million MNT")
print(f"  RMSE: {rmse_rf:.2f} million MNT")

In [ ]:
# Train XGBoost
print("\nTraining XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=10, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate
r2_xgb = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f"\nXGBoost Results:")
print(f"  R² Score: {r2_xgb:.4f} ({r2_xgb*100:.2f}% accuracy)")
print(f"  MAE: {mae_xgb:.2f} million MNT")
print(f"  RMSE: {rmse_xgb:.2f} million MNT")

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest
axes[0].scatter(y_test, y_pred_rf, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price (million MNT)')
axes[0].set_ylabel('Predicted Price (million MNT)')
axes[0].set_title(f'Random Forest (R² = {r2_rf:.4f})')
axes[0].grid(True, alpha=0.3)

# XGBoost
axes[1].scatter(y_test, y_pred_xgb, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Price (million MNT)')
axes[1].set_ylabel('Predicted Price (million MNT)')
axes[1].set_title(f'XGBoost (R² = {r2_xgb:.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
- ✓ Loading 12 months of data
- ✓ Deduplication across files
- ✓ Data cleaning and outlier removal
- ✓ Feature engineering
- ✓ Answering all 11 research questions
- ✓ Training ML models for price prediction

**Next Steps:**
1. Hyperparameter tuning for better accuracy
2. Try advanced models (LightGBM, CatBoost)
3. Feature selection and engineering
4. Ensemble methods
5. Deploy as web application